In [1]:
import numpy as np
import scipy.signal as signal
import mne


In [3]:
import numpy as np
import scipy.signal as signal

# -----------------------------
# LOAD DEAP DATA
# -----------------------------
def load_deap_data(file_path):
    data = np.load(file_path, allow_pickle=True, encoding='latin1')
    return data['data'], data['labels']


# -----------------------------
# PREPROCESSING
# -----------------------------
def bandpass_filter(eeg_signal, low=0.5, high=40, fs=128):
    nyquist = 0.5 * fs
    b, a = signal.butter(4, [low/nyquist, high/nyquist], btype='band')
    return signal.filtfilt(b, a, eeg_signal, axis=-1)

def normalize_signal(eeg_signal):
    mean = np.mean(eeg_signal, axis=-1, keepdims=True)
    std = np.std(eeg_signal, axis=-1, keepdims=True)
    return (eeg_signal - mean) / (std + 1e-6)

def preprocess_eeg(eeg_data):
    processed = []
    for trial in eeg_data:
        filtered = bandpass_filter(trial)
        normalized = normalize_signal(filtered)
        processed.append(normalized)
    return np.array(processed)


# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def time_domain_features(eeg_window):
    features = []
    for ch in eeg_window:
        features.extend([np.mean(ch), np.std(ch), np.var(ch)])
    return features

def bandpower(data, fs, band):
    freqs, psd = signal.welch(data, fs=fs)
    idx = (freqs >= band[0]) & (freqs <= band[1])
    return np.sum(psd[idx])

def frequency_features(eeg_window, fs=128):
    features = []
    bands = [(0.5,4), (4,8), (8,13), (13,30)]
    
    for ch in eeg_window:
        for band in bands:
            features.append(bandpower(ch, fs, band))
    return features

def extract_features(eeg_window):
    return np.array(
        time_domain_features(eeg_window) +
        frequency_features(eeg_window)
    )


# -----------------------------
# WINDOWING
# -----------------------------
def create_windows(eeg_trial, window_size=256, overlap=128):
    windows = []
    start = 0
    
    while start + window_size <= eeg_trial.shape[1]:
        windows.append(eeg_trial[:, start:start + window_size])
        start += overlap
        
    return windows


# -----------------------------
# BUILD DATASET
# -----------------------------
def build_feature_dataset(eeg_data, labels):
    X, y = [], []

    for i in range(len(eeg_data)):
        windows = create_windows(eeg_data[i])
        
        for w in windows:
            X.append(extract_features(w))
            y.append(labels[i])  # raw labels for now

    return np.array(X), np.array(y)


# -----------------------------
# RUN
# -----------------------------
file_path = r"E:\Downloads\archive (6)\deap-dataset\data_preprocessed_python\s01.dat"

eeg, labels = load_deap_data(file_path)

print("Original:", eeg.shape)

eeg = preprocess_eeg(eeg)

X, y = build_feature_dataset(eeg, labels)

print("Features:", X.shape)
print("Labels:", y.shape)

Original: (40, 40, 8064)
Features: (2480, 280)
Labels: (2480, 4)


In [4]:
def map_emotion(label):
    valence = label[0]
    arousal = label[1]

    # Remove unclear samples
    if 4 <= valence <= 6 or 4 <= arousal <= 6:
        return None

    if valence > 6 and arousal > 6:
        return "Excited"
    elif valence > 6 and arousal < 4:
        return "Calm"
    elif valence < 4 and arousal > 6:
        return "Stressed"
    elif valence < 4 and arousal < 4:
        return "Sad"
    else:
        return "Neutral"

In [5]:
new_X = []
new_y = []

for i in range(len(y)):
    emotion = map_emotion(y[i])
    
    if emotion is not None:
        new_X.append(X[i])
        new_y.append(emotion)

new_X = np.array(new_X)
new_y = np.array(new_y)

print("After filtering:", new_X.shape, new_y.shape)

After filtering: (2232, 280) (2232,)


In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(new_y)

print("Classes:", le.classes_)

Classes: ['Calm' 'Sad' 'Stressed']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    new_X, y_encoded, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [9]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.6487695749440716

Report:
               precision    recall  f1-score   support

        Calm       1.00      0.03      0.07        88
         Sad       0.64      0.96      0.76       244
    Stressed       0.70      0.46      0.55       115

    accuracy                           0.65       447
   macro avg       0.78      0.48      0.46       447
weighted avg       0.72      0.65      0.57       447



In [10]:
import joblib

joblib.dump(model, "emotion_model.pkl")
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [11]:
import joblib

model = joblib.load("emotion_model.pkl")
le = joblib.load("label_encoder.pkl")

In [12]:
trial = eeg[0]   # shape: (channels, samples)

In [13]:
window_size = 256
step = 128  # overlap

predictions = []

start = 0

while start + window_size <= trial.shape[1]:
    
    window = trial[:, start:start + window_size]
    
    features = extract_features(window).reshape(1, -1)
    
    pred = model.predict(features)[0]
    prob = model.predict_proba(features)[0]
    
    emotion = le.inverse_transform([pred])[0]
    
    predictions.append(emotion)
    
    print(f"Window {start}: {emotion} ({max(prob):.2f})")
    
    start += step

Window 0: Sad (0.78)
Window 128: Sad (0.90)
Window 256: Sad (0.85)
Window 384: Sad (0.79)
Window 512: Sad (0.87)
Window 640: Sad (0.86)
Window 768: Sad (0.80)
Window 896: Sad (0.85)
Window 1024: Sad (0.83)
Window 1152: Sad (0.81)
Window 1280: Sad (0.78)
Window 1408: Sad (0.86)
Window 1536: Sad (0.80)
Window 1664: Sad (0.47)
Window 1792: Sad (0.84)
Window 1920: Sad (0.86)
Window 2048: Sad (0.83)
Window 2176: Sad (0.89)
Window 2304: Sad (0.65)
Window 2432: Sad (0.90)
Window 2560: Sad (0.79)
Window 2688: Sad (0.85)
Window 2816: Sad (0.83)
Window 2944: Sad (0.78)
Window 3072: Sad (0.83)
Window 3200: Sad (0.63)
Window 3328: Sad (0.86)
Window 3456: Sad (0.82)
Window 3584: Sad (0.79)
Window 3712: Sad (0.59)
Window 3840: Sad (0.83)
Window 3968: Sad (0.90)
Window 4096: Sad (0.76)
Window 4224: Sad (0.88)
Window 4352: Sad (0.83)
Window 4480: Sad (0.85)
Window 4608: Sad (0.90)
Window 4736: Sad (0.87)
Window 4864: Sad (0.88)
Window 4992: Sad (0.82)
Window 5120: Sad (0.91)
Window 5248: Sad (0.92)
Wi

In [14]:
from collections import Counter

def smooth_predictions(preds, window=5):
    smoothed = []
    
    for i in range(len(preds)):
        chunk = preds[max(0, i-window):i+1]
        most_common = Counter(chunk).most_common(1)[0][0]
        smoothed.append(most_common)
    
    return smoothed


smooth_preds = smooth_predictions(predictions)

In [15]:
print("\nFinal Emotion:", Counter(smooth_preds).most_common(1)[0][0])


Final Emotion: Sad


In [16]:
model = RandomForestClassifier(n_estimators=100, class_weight='balanced')

In [17]:
import time
time.sleep(0.2)

In [19]:
!pip install streamlit

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ----- ---------------------------------- 1.3/9.1 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------  8.9/9.1 MB 29.1 MB/s eta 0:00:01
   ---------------------------------------- 9.1/9.1 MB 26.8 MB/s  0:00:00
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/795.4 kB ? eta -:--:--
   ------------- -------------------------- 262.1/795.4 kB ? eta -:--:--
   ------------- -------------------------- 262.1/795.4 kB ? eta -:--:--
   ------------------------- ------------ 524.3/795.4 kB 882.6 kB/s eta 0:00:01
   ---------------------------------------- 795.4/795.4 kB 1.1 MB/s  0:00:01

   ---------- ----------------------------- 1/4 [gitpython]
   -------------------- ------------------- 2/4 [altair

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python311\\Scripts\\streamlit.exe' -> 'C:\\Python311\\Scripts\\streamlit.exe.deleteme'


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import streamlit as st

st.title("EEG Emotion Recognition")

st.write("Current Emotion:", smooth_preds[-1])

st.line_chart(smooth_preds)

2026-04-08 18:23:08.516 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 18:23:08.635 
  command:

    streamlit run C:\Users\Aakif Khan\AppData\Roaming\Python\Python311\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-08 18:23:08.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 18:23:08.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 18:23:08.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 18:23:08.645 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 18:23:08.646 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 18:23:08.647

DeltaGenerator()

In [ ]:
import joblib

joblib.dump(model, "emotion_model.pkl")
joblib.dump(le, "label_encoder.pkl")